# Filtrage JSTOR — Poesie americaine
### Memoire M1 HN — Ecole des Chartes / PSL

Execute les cellules dans l'ordre avec **Shift + Entree**.

## Etape 0 — Installation

In [1]:
!pip install tqdm -q
print("OK")

OK


## Etape 1 — Nom du fichier
Modifie uniquement la ligne NOM_DU_FICHIER.

In [2]:
import os

# MODIFIE CETTE LIGNE
NOM_DU_FICHIER = "jstor_metadata_2026-05-03.jsonl.gz"

if os.path.exists(NOM_DU_FICHIER):
    taille = os.path.getsize(NOM_DU_FICHIER) / (1024 * 1024)
    print("Fichier trouve ! (" + str(round(taille, 1)) + " Mo)")
else:
    print("ERREUR : fichier introuvable -> " + NOM_DU_FICHIER)

Fichier trouve ! (1232.1 Mo)


## Etape 2 — Configuration
Rien a modifier.

**Note** : les noms de revues sont les graphies exactes trouvees dans le champ `is_part_of` du fichier JSTOR.
Le filtre est strict : un article doit appartenir exactement a l'une de ces revues.

In [3]:
# Graphies exactes verifiees dans is_part_of
REVUES_CIBLEES = [
    "poetry",
    "the american poetry review",
    "the kenyon review",
    "prairie schooner",
]
ANNEE_MIN = 1900
FICHIER_SORTIE = "ids_poesie_americaine.txt"
print("Configuration OK")
print("Revues ciblees :")
for r in REVUES_CIBLEES:
    print("  - " + r)

Configuration OK
Revues ciblees :
  - poetry
  - the american poetry review
  - the kenyon review
  - prairie schooner


## Etape 3 — Filtrage

Filtre les articles (`content_type == article`) dont le champ `is_part_of` correspond
**exactement** a l'une des revues ciblees.

Correction par rapport a la version precedente : on ne cherche plus le mot dans tous les champs
(ce qui captait des faux positifs comme *Paideuma* ou *Poetry Foundation*),
mais on compare uniquement le champ `is_part_of` avec correspondance exacte.

In [4]:
import gzip, json, os
from tqdm import tqdm

item_ids = []
compteur_trouves = 0
revues_trouvees = {}

taille_fichier = os.path.getsize(NOM_DU_FICHIER)
print("Debut du filtrage...")
print("(Filtre strict : correspondance exacte sur is_part_of)")
print("")

with gzip.open(NOM_DU_FICHIER, "rb") as f_raw:
    with tqdm(total=taille_fichier, unit="o", unit_scale=True, desc="Lecture", mininterval=2.0) as barre:
        position_precedente = 0
        for ligne in f_raw:
            position_actuelle = f_raw.fileobj.tell()
            barre.update(position_actuelle - position_precedente)
            position_precedente = position_actuelle

            try:
                data = json.loads(ligne.decode("utf-8"))
            except:
                continue

            # Filtre 1 : uniquement les articles
            if data.get("content_type") != "article":
                continue

            # Filtre 2 : correspondance EXACTE sur is_part_of uniquement
            is_part_of = data.get("is_part_of", "")
            if not is_part_of:
                continue
            valeur_min = str(is_part_of).lower().strip()
            if valeur_min not in REVUES_CIBLEES:
                continue
            nom_revue = str(is_part_of).strip()

            # Filtre 3 : verifier l'annee
            annee = (data.get("pub_year") or data.get("year") or
                     data.get("date") or data.get("published_date") or
                     data.get("publication_year"))
            annee_ok = False
            if annee:
                try:
                    annee_ok = int(str(annee)[:4]) >= ANNEE_MIN
                except:
                    pass
            if not annee_ok:
                continue

            # Recuperer l'ID
            item_id = (data.get("item_id") or data.get("id") or
                       data.get("doi") or data.get("ithaka_doi"))
            if item_id:
                item_ids.append(str(item_id))
                compteur_trouves += 1
                revues_trouvees[nom_revue] = revues_trouvees.get(nom_revue, 0) + 1
                if compteur_trouves % 500 == 0:
                    barre.set_postfix({"trouves": compteur_trouves})

print("")
print("Filtrage termine !")
print(str(compteur_trouves) + " articles trouves")
if revues_trouvees:
    print("")
    print("Repartition par revue :")
    for revue, nb in sorted(revues_trouvees.items(), key=lambda x: -x[1]):
        print("  " + revue + " : " + str(nb) + " articles")

Debut du filtrage...
(Filtre strict : correspondance exacte sur is_part_of)



Lecture: 100%|██████████████| 1.29G/1.29G [04:47<00:00, 4.49Mo/s, trouves=89000]


Filtrage termine !
89470 articles trouves

Repartition par revue :
  Poetry : 45736 articles
  Prairie Schooner : 18257 articles
  The American Poetry Review : 15875 articles
  The Kenyon Review : 9602 articles


## Etape 4 — Sauvegarde

In [5]:
if item_ids:
    with open(FICHIER_SORTIE, "w", encoding="utf-8") as f_out:
        for item_id in item_ids:
            f_out.write(item_id + "\n")
    print("Fichier cree : " + FICHIER_SORTIE)
    print(str(len(item_ids)) + " IDs enregistres")
    print("")
    print("Prochaine etape :")
    print("1. Va sur https://jstor.org/ta-support/metadata")
    print("2. Clique Create request")
    print("3. Remplis le formulaire (master, usage academique)")
    print("4. Upload TXT -> " + FICHIER_SORTIE)
    print("5. Soumets et attends l'email JSTOR")
else:
    print("Aucun article trouve.")

Fichier cree : ids_poesie_americaine.txt
89470 IDs enregistres

Prochaine etape :
1. Va sur https://jstor.org/ta-support/metadata
2. Clique Create request
3. Remplis le formulaire (master, usage academique)
4. Upload TXT -> ids_poesie_americaine.txt
5. Soumets et attends l'email JSTOR


## Bonus — Verification
Verifie les 10 premiers IDs avant de soumettre a JSTOR.

In [6]:
if item_ids:
    print("Les 10 premiers IDs trouves :")
    for i, item_id in enumerate(item_ids[:10], 1):
        print(str(i) + ". " + item_id)
    print("... et " + str(len(item_ids) - 10) + " autres.")
    print("")
    print("Verifie un ID sur : https://www.jstor.org/stable/[ID]")
else:
    print("Aucun article a afficher.")

Les 10 premiers IDs trouves :
1. 7c643338-2a1d-3f0e-aca1-9ada3e520309
2. 31a9213c-d25b-399d-8f48-71ce1f6e369b
3. 9809783e-9aca-3ddb-bf6c-226915ce8f25
4. cd9b957c-beb7-3de8-9ac2-ad709b35157b
5. 97da565d-faef-32ae-a893-f39d876777ee
6. 44d8a403-0744-36f6-bd2f-62237ee00983
7. af6f2b4d-f16b-3b94-a36c-f50b75ac4371
8. 7cbdd743-b490-3b29-9f50-35c308d4818f
9. 613cf46f-6158-3f4f-9d55-698facda926e
10. e2df5559-0738-3032-a578-60971f746393
... et 89460 autres.

Verifie un ID sur : https://www.jstor.org/stable/[ID]
